# grok-011 · 农作物病害图像分类（学习向）

> **系列转向**：从 001–010 的 LLM/T4 路线，进入 **计算机视觉 · 图像分类**。
>
> **任务**：输入一张作物叶片照片，判断它属于 **38 类** 病害/健康类别中的哪一类（PlantVillage 系数据）。
>
> **你在学什么**
> 1. 真实图像分类流水线：路径发现 → 划分/增强 → DataLoader → 迁移学习 → 指标
> 2. **ImageNet 预训练 + 换头**（EfficientNet-B0），而不是从零训大网
> 3. 类别不均衡时的读数：Top-1、macro-F1、混淆矩阵（别只看 overall accuracy）
> 4. 农业场景坑：受控背景数据集 vs 田间照片的 **域偏移**（本课数据偏「实验室拍照」）
>
> **不学什么**：目标检测、分割、自监督预训练、部署端侧量化（后续 CV 课再开）。

## 建议阅读顺序
1. 确认 GPU + 找到数据根目录  
2. 看类别分布（哪些作物/病多）  
3. 迁移学习训练几个 epoch  
4. 冻结 eval 集上看 Top-1 / macro-F1 / 最易混的类对  


# grok-011-crop-disease-cls

> **Slug/title:** `grok-011-crop-disease-cls` — Crop leaf disease classification (PlantVillage / New Plant Diseases)

## 本 notebook 做什么
1. 自动发现 Kaggle Input 中的 **New Plant Diseases / PlantVillage** 目录结构  
2. `torchvision` **EfficientNet-B0** ImageNet 预训练，替换分类头 → 38 类  
3. 训练时增强（RandomResizedCrop / Flip / ColorJitter）；验证时确定性 resize+center crop  
4. AMP（fp16）在 T4 上提速；记录 peak VRAM  
5. 写出 `grok011_results.json`、`best.pt`、`class_names.json`、混淆矩阵图  

## 数据（二选一）
| 方式 | 说明 |
|---|---|
| **推荐** | Kaggle 右侧 **Add Data** → 搜 `New Plant Diseases Dataset`（vipoooool） |
| 联网备用 | `kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")` |

> 完整约 **87k** 张、**38** 类。本课默认 `FAST_MODE=True`：每类最多抽 N 张，保证一节 T4 配额内跑完；关掉即可全量。

## 与 001 的关系
- 001 是合成小 CNN **冒烟**；011 是 **真数据 + 迁移学习 + 正经指标** 的分类课。


In [ ]:
# 【步骤】环境 + GPU
import os, re, json, time, random, platform, math, traceback
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torch.cuda.amp import autocast, GradScaler

print("torch", torch.__version__)
print("python", platform.python_version())
print("cuda_available", torch.cuda.is_available())
assert torch.cuda.is_available(), "请在 Kaggle 打开 GPU（Accelerator = GPU T4）"
print("device_count", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}", torch.cuda.get_device_name(i),
          f"mem={p.total_memory/1e9:.1f}GB")
DEVICE = torch.device("cuda:0")
# 本课单卡足够；双 T4 时仍用 cuda:0，避免 DataParallel 把小 batch 搞复杂
torch.backends.cudnn.benchmark = True


In [ ]:
# 【步骤】超参：FAST_MODE 先跑通，再关全量
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# True：每类最多 MAX_PER_CLASS 张，约 10–20 min 级；False：全量 train/valid
FAST_MODE = True
MAX_PER_CLASS = 120          # FAST 时每类训练样本上限（valid 另有目录则不截断过多）
MAX_PER_CLASS_VAL = 40

IMG_SIZE = 224
BATCH_SIZE = 64              # T4 16GB + EfficientNet-B0 很宽裕
NUM_WORKERS = 2
EPOCHS = 5 if FAST_MODE else 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
NUM_CLASSES_EXPECT = 38      # PlantVillage 增强集常见 38 类

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working_011")
OUT.mkdir(parents=True, exist_ok=True)
print("OUT =", OUT.resolve())
print("FAST_MODE", FAST_MODE, "EPOCHS", EPOCHS, "BATCH", BATCH_SIZE)


In [ ]:
# 【步骤】定位数据集根目录（Kaggle Input / kagglehub / 本地）
from torchvision import datasets, transforms, models

def find_imagefolder_roots(start: Path):
    """在 start 下找包含多个子文件夹、且子文件夹内有图片的 train/valid 根。"""
    hits = []
    if not start.exists():
        return hits
    for p in start.rglob("*"):
        if not p.is_dir():
            continue
        name = p.name.lower()
        if name in {"train", "train set", "training"}:
            # 至少 5 个子类目录
            subs = [d for d in p.iterdir() if d.is_dir()]
            if len(subs) >= 5:
                hits.append(p)
    return hits


def pair_valid(train_dir: Path):
    """同级或父级找 valid/val/test。"""
    cands = []
    for parent in [train_dir.parent, train_dir.parent.parent]:
        for nm in ["valid", "validation", "val", "test"]:
            c = parent / nm
            if c.is_dir() and any(c.iterdir()):
                cands.append(c)
            # 有时 valid 叫 Valid
            for d in parent.iterdir() if parent.exists() else []:
                if d.is_dir() and d.name.lower() == nm:
                    cands.append(d)
    # 去重
    uniq, seen = [], set()
    for c in cands:
        rp = c.resolve()
        if rp not in seen:
            seen.add(rp)
            uniq.append(c)
    return uniq[0] if uniq else None


SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/input/new-plant-diseases-dataset"),
    Path("./data"),
    Path("/workspace/kaggle-lab/data"),
]

train_dir = None
for root in SEARCH_ROOTS:
    for t in find_imagefolder_roots(root):
        train_dir = t
        break
    if train_dir:
        break

if train_dir is None:
    # 联网备用：kagglehub
    try:
        import kagglehub
        print("local not found → kagglehub download vipoooool/new-plant-diseases-dataset ...")
        path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
        print("downloaded to", path)
        for t in find_imagefolder_roots(Path(path)):
            train_dir = t
            break
    except Exception as e:
        print("kagglehub failed:", type(e).__name__, e)

assert train_dir is not None, (
    "未找到数据。请在 Kaggle Add Data → 'New Plant Diseases Dataset' (vipoooool)，"
    "或设置联网后重跑本 cell。"
)
val_dir = pair_valid(train_dir)
print("train_dir =", train_dir)
print("val_dir   =", val_dir)


In [ ]:
# 【步骤】构建 Dataset；FAST_MODE 下分层截断
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = datasets.ImageFolder(str(train_dir), transform=train_tf)
class_names = list(full_train.classes)
n_classes = len(class_names)
print("n_classes", n_classes, "(expect ~", NUM_CLASSES_EXPECT, ")")
print("n_train_raw", len(full_train))
print("sample classes:", class_names[:5], "...")

# 类别计数
train_targets = full_train.targets  # list[int]
cnt = Counter(train_targets)
print("per-class min/max", min(cnt.values()), max(cnt.values()))


def subset_by_class(dataset, max_per, seed=SEED):
    rng = random.Random(seed)
    buckets = defaultdict(list)
    # ImageFolder samples: (path, label)
    for idx, y in enumerate(dataset.targets):
        buckets[y].append(idx)
    keep = []
    for y, idxs in buckets.items():
        rng.shuffle(idxs)
        keep.extend(idxs[:max_per] if max_per else idxs)
    rng.shuffle(keep)
    return Subset(dataset, keep)


if FAST_MODE:
    train_ds = subset_by_class(full_train, MAX_PER_CLASS)
else:
    train_ds = full_train

if val_dir is not None:
    full_val = datasets.ImageFolder(str(val_dir), transform=eval_tf)
    # 对齐 class_to_idx
    assert full_val.class_to_idx == full_train.class_to_idx, "train/valid 类别名不一致，请检查目录"
    if FAST_MODE:
        val_ds = subset_by_class(full_val, MAX_PER_CLASS_VAL, seed=SEED + 1)
    else:
        val_ds = full_val
else:
    # 无 valid 目录：从 train 划 15%
    print("no valid dir → split 15% from train")
    full_eval = datasets.ImageFolder(str(train_dir), transform=eval_tf)
    idxs = list(range(len(full_train)))
    random.Random(SEED).shuffle(idxs)
    n_val = max(1, int(0.15 * len(idxs)))
    val_idx, tr_idx = idxs[:n_val], idxs[n_val:]
    if FAST_MODE:
        # 在已截断集合上再拆较乱，改为按类从 raw 抽
        train_ds = subset_by_class(full_train, MAX_PER_CLASS)
        # val 用 eval transform 的平行 Subset
        val_base = full_eval
        val_ds = subset_by_class(val_base, MAX_PER_CLASS_VAL, seed=SEED + 1)
    else:
        train_ds = Subset(full_train, tr_idx)
        val_ds = Subset(full_eval, val_idx)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print("train batches", len(train_loader), "samples", len(train_ds))
print("val   batches", len(val_loader), "samples", len(val_ds))


In [ ]:
# 【步骤】快速 EDA：类别分布 + 类名结构（作物___病害）
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 统计当前 train_ds 标签
ys = []
if isinstance(train_ds, Subset):
    base = train_ds.dataset
    for i in train_ds.indices:
        ys.append(base.targets[i])
else:
    ys = list(train_ds.targets)
ycnt = Counter(ys)

# 解析 PlantVillage 命名: Apple___Apple_scab
crops = Counter()
for name in class_names:
    crop = name.split("___")[0] if "___" in name else name.split("_")[0]
    crops[crop] += 1

print("作物种类数（粗分）", len(crops))
print("top crops by #disease-classes:", crops.most_common(10))

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(n_classes), [ycnt.get(i, 0) for i in range(n_classes)], color="#2a9d8f")
ax.set_title("Train samples per class (after FAST subsample)" if FAST_MODE else "Train samples per class")
ax.set_xlabel("class id")
ax.set_ylabel("count")
fig.tight_layout()
fig.savefig(OUT / "class_hist.png", dpi=120)
plt.close(fig)
print("saved", OUT / "class_hist.png")

# 写类名表
(OUT / "class_names.json").write_text(
    json.dumps({"classes": class_names, "class_to_idx": full_train.class_to_idx}, ensure_ascii=False, indent=2)
)


In [ ]:
# 【步骤】模型：EfficientNet-B0 预训练 + 新分类头
# 若离线无权重，weights=None 仍可训，但收敛差很多 → Kaggle 请开 Internet 或预先加模型数据集

def build_model(n_classes: int):
    try:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        m = models.efficientnet_b0(weights=weights)
        print("loaded ImageNet pretrained EfficientNet-B0")
    except Exception as e:
        print("pretrained load failed, random init:", e)
        m = models.efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_f, n_classes)
    return m

model = build_model(n_classes).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params {n_params/1e6:.2f}M  trainable {n_train/1e6:.2f}M")

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# 余弦退火：分类微调常用
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler(enabled=True)


In [ ]:
# 【步骤】训练 + 验证（Top-1 / macro-F1）
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    all_y, all_p = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast(enabled=True):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            n += x.size(0)
            all_y.append(y.cpu())
            all_p.append(pred.cpu())
    y_true = torch.cat(all_y).numpy()
    y_pred = torch.cat(all_p).numpy()
    acc = correct / max(1, n)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return {
        "loss": total_loss / max(1, n),
        "acc": acc,
        "macro_f1": float(f1),
        "y_true": y_true,
        "y_pred": y_pred,
    }

history = []
best = {"acc": -1.0, "epoch": -1}
t0 = time.perf_counter()
torch.cuda.reset_peak_memory_stats()

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    scheduler.step()
    row = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": tr["loss"],
        "train_acc": tr["acc"],
        "val_loss": va["loss"],
        "val_acc": va["acc"],
        "val_macro_f1": va["macro_f1"],
    }
    history.append(row)
    print(
        f"epoch {epoch:02d}/{EPOCHS}  "
        f"tr_loss={tr['loss']:.4f} tr_acc={tr['acc']:.3f}  "
        f"va_loss={va['loss']:.4f} va_acc={va['acc']:.3f} va_f1={va['macro_f1']:.3f}  "
        f"lr={row['lr']:.2e}"
    )
    if va["acc"] > best["acc"]:
        best = {"acc": va["acc"], "epoch": epoch, "macro_f1": va["macro_f1"]}
        torch.save({
            "model": model.state_dict(),
            "n_classes": n_classes,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "epoch": epoch,
            "val_acc": va["acc"],
            "val_macro_f1": va["macro_f1"],
        }, OUT / "best.pt")
        best_y_true, best_y_pred = va["y_true"], va["y_pred"]

train_seconds = time.perf_counter() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"done in {train_seconds:.1f}s  peak_vram={peak_gb:.2f}GB  best_val_acc={best['acc']:.3f}@ep{best['epoch']}")


In [ ]:
# 【步骤】混淆矩阵 + 最易混淆的类对 + 分类报告摘要
cm = confusion_matrix(best_y_true, best_y_pred, labels=list(range(n_classes)))

# 非对角最大的若干对
pairs = []
for i in range(n_classes):
    for j in range(n_classes):
        if i == j:
            continue
        if cm[i, j] > 0:
            pairs.append((int(cm[i, j]), class_names[i], class_names[j]))
pairs.sort(reverse=True)
print("Top confused (true → pred):")
for c, a, b in pairs[:12]:
    print(f"  {c:4d}  {a}  →  {b}")

# 画缩小版混淆矩阵（类多，用 log1p 着色）
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(np.log1p(cm), interpolation="nearest", cmap="Blues")
ax.set_title("Confusion matrix (log1p scale)")
ax.set_xlabel("pred")
ax.set_ylabel("true")
fig.tight_layout()
fig.savefig(OUT / "confusion_matrix.png", dpi=140)
plt.close(fig)

report = classification_report(
    best_y_true, best_y_pred,
    target_names=class_names,
    zero_division=0,
    digits=3,
)
(OUT / "classification_report.txt").write_text(report)
print(report[:1500], "\n...")


In [ ]:
# 【步骤】推理探头：随机抽验证集几张看预测
import torchvision.transforms.functional as TF
from PIL import Image

# 取未归一化路径样本更直观：从 ImageFolder samples 读原图
base_val = val_ds.dataset if isinstance(val_ds, Subset) else val_ds
idxs = list(val_ds.indices) if isinstance(val_ds, Subset) else list(range(len(val_ds)))
random.Random(0).shuffle(idxs)
show_n = min(6, len(idxs))

model.eval()
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
axes = axes.ravel()
for ax, idx in zip(axes, idxs[:show_n]):
    path, y = base_val.samples[idx]
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), autocast(enabled=True):
        prob = model(x).softmax(1)[0]
    conf, pred = prob.max(0)
    pred_i, conf = int(pred), float(conf)
    ok = pred_i == y
    ax.imshow(img)
    ax.set_title(
        f"{'OK' if ok else 'ERR'} {conf:.2f}\n"
        f"T:{class_names[y].split('___')[-1][:22]}\n"
        f"P:{class_names[pred_i].split('___')[-1][:22]}",
        fontsize=8,
        color=("green" if ok else "crimson"),
    )
    ax.axis("off")
fig.suptitle("Val samples: true vs pred")
fig.tight_layout()
fig.savefig(OUT / "samples_pred.png", dpi=130)
plt.close(fig)
print("saved", OUT / "samples_pred.png")


In [ ]:
# 【步骤】持久化结果 JSON（方便和后续 012 检测课对齐风格）
results = {
    "notebook": "grok-011-crop-disease-cls",
    "task": "crop_disease_image_classification",
    "dataset": {
        "train_dir": str(train_dir),
        "val_dir": str(val_dir) if val_dir else None,
        "n_classes": n_classes,
        "class_names": class_names,
        "fast_mode": FAST_MODE,
        "max_per_class": MAX_PER_CLASS if FAST_MODE else None,
        "n_train": len(train_ds),
        "n_val": len(val_ds),
    },
    "model": "torchvision.efficientnet_b0",
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "history": [
        {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in row.items()}
        for row in history
    ],
    "best": best,
    "train_seconds": train_seconds,
    "peak_vram_gb": peak_gb,
    "device": torch.cuda.get_device_name(0),
    "artifacts": [
        "best.pt",
        "class_names.json",
        "class_hist.png",
        "confusion_matrix.png",
        "classification_report.txt",
        "samples_pred.png",
        "grok011_results.json",
    ],
}
(OUT / "grok011_results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
print(json.dumps({k: results[k] for k in ["best", "train_seconds", "peak_vram_gb", "dataset"]}, indent=2, ensure_ascii=False)[:1200])
print("\nall written under", OUT)


## 学习检查清单

- 你应能回答：
  1. 为什么用 ImageNet 预训练，而不是随机初始化 EfficientNet？
  2. `train` 与 `eval` 的 transform 为何不同？少了增强会怎样？
  3. **macro-F1** 和 **accuracy** 在类别不均衡时谁更「诚实」？
  4. 混淆矩阵里高频 `true→pred` 对说明了什么（相似病斑 / 同作物不同病）？
  5. PlantVillage 受控背景 vs 真实田间照片：上线前还缺什么？
- 建议动手：
  - 把 `FAST_MODE=False` 跑全量，看 val_acc 变化  
  - 换成 `resnet18` 对比收敛与 peak VRAM  
  - 只保留 1 种作物的子集，看是否更容易过拟合  

## 产物
| 文件 | 含义 |
|---|---|
| `best.pt` | 验证集最优权重 + 类名 |
| `grok011_results.json` | 指标与超参 |
| `confusion_matrix.png` | 混淆矩阵 |
| `samples_pred.png` | 可视化对错 |
